In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# src 경로 등록 + config 로드
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bootstrap import setup_notebook
ROOT, cfg, _vars = setup_notebook("dog.yaml")
globals().update(_vars)

import os
import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import StableDiffusionPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
if "pipe" not in globals():
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
    pipe = pipe.to(DEVICE)
    pipe.safety_checker = None
    pipe.set_progress_bar_config(disable=True)


In [ ]:
from load import load_cache
_cache = load_cache(cfg, "neuron_selection")
for k, v in _cache.items():
    globals()[k] = v
if hasattr(dino_embeds, 'numpy'):
    dino_embeds = dino_embeds.numpy()


In [ ]:
from load import load_images_and_ff_in
if "images" not in globals() or images is None:
    images, _, _ = load_images_and_ff_in(cfg)


# multi-plane

In [ ]:
# ── 다중 hyperplane (3개) — STEER_PLANES 수동 지정 ───────────────────────────
# sel / print(sel) 로 후보 확인 후, 아래 리스트만 원하는 뉴런 index 3개로 수정
import numpy as np
import torch

STEER_PLANES = [sel[0], sel[1], sel[2]]   # ← 임의 3개 (neuron index)

# ① INJECT_BLOCK 출력 활성값 로드
inject_list = []
for p_idx, prompt in enumerate(PROMPTS):
    pdir = resolve_pdir(p_idx, prompt)
    n    = NUM_IMAGES_PER_PROMPT
    data = torch.load(pdir / INJECT_BLOCK.replace(".", "_") / "activations.pt",
                      weights_only=True)
    inject_list.append(data["outputs"][:n].cpu().float())
inject_out = torch.cat(inject_list, dim=0)            # (N, C, H, W)
print(f"STEER_PLANES: {STEER_PLANES}")
print(f"inject_out: {tuple(inject_out.shape)}")

plane_signs = binary[:, STEER_PLANES].cpu().numpy()   # (N, 3), 0/1

In [ ]:
sel

In [ ]:
# 8 구역 대표 이미지 — n≥5 region만 표시
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgridspec

MIN_REGION_VIZ = 5
region_ids = plane_signs[:, 0] * 4 + plane_signs[:, 1] * 2 + plane_signs[:, 2] * 1
N_REGIONS = 8
N_SHOW_PER_REGION = 4

region_counts = [int(np.sum(region_ids == r)) for r in range(N_REGIONS)]
active_regions = [r for r in range(N_REGIONS) if region_counts[r] >= MIN_REGION_VIZ]
skipped_regions = [r for r in range(N_REGIONS) if region_counts[r] < MIN_REGION_VIZ]

if not active_regions:
    print(f"n≥{MIN_REGION_VIZ} region 없음")
else:
    NROWS = len(active_regions)
    NCOLS = N_SHOW_PER_REGION
    CELL_W = 2.75
    CELL_H = 2.65
    LABEL_W = 0.95

    FIG_BG  = '#F5F6F8'
    PANEL   = '#FFFFFF'
    BORDER  = '#C4C9D1'
    TEXT    = '#1A1A2E'
    MUTED   = '#5F6478'

    fig = plt.figure(
        figsize=(LABEL_W + NCOLS * CELL_W, NROWS * CELL_H + 0.35),
        facecolor=FIG_BG,
    )
    gs = mgridspec.GridSpec(
        NROWS, NCOLS + 1, figure=fig,
        width_ratios=[LABEL_W / CELL_W] + [1.0] * NCOLS,
        wspace=0.012, hspace=0.08,
        left=0.02, right=0.995, top=0.92, bottom=0.02,
    )

    for row, region in enumerate(active_regions):
        bits = f"{(region >> 2) & 1}{(region >> 1) & 1}{region & 1}"
        idxs = np.where(region_ids == region)[0]
        chosen = idxs[:N_SHOW_PER_REGION]

        ax_lbl = fig.add_subplot(gs[row, 0])
        ax_lbl.set_facecolor(FIG_BG)
        ax_lbl.axis('off')
        ax_lbl.text(
            0.95, 0.5,
            f"R{region}\n[{bits}]\nn={region_counts[region]}",
            ha='right', va='center', fontsize=9.5, fontweight='bold',
            fontfamily='monospace', color=TEXT,
            transform=ax_lbl.transAxes,
        )

        for k in range(NCOLS):
            ax = fig.add_subplot(gs[row, k + 1])
            ax.set_facecolor(PANEL)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_color(BORDER)
                spine.set_linewidth(0.7)

            if k < len(chosen):
                i = chosen[k]
                ax.imshow(np.array(images[i]))
                ax.set_title(f"#{i}", fontsize=8.5, pad=2, color=MUTED)
            else:
                ax.axis('off')

    # skipped_txt = ', '.join(
    #     f"R{r}[{(r>>2)&1}{(r>>1)&1}{r&1}]={region_counts[r]}"
    #     for r in skipped_regions
    # )
    fig.suptitle(
        f"Regions with n≥{MIN_REGION_VIZ}  ({N_SHOW_PER_REGION} samples each)",
        fontsize=12, fontweight='bold', color=TEXT, y=0.985,
    )
    # if skipped_txt:
        # fig.text(
        #     0.5, 0.955,
        #     f"shown {len(active_regions)}/{N_REGIONS}  ·  skipped: {skipped_txt}",
        #     ha='center', fontsize=8.5, color=MUTED,
        # )
    plt.show()

In [ ]:
# ② region별 이미지 인덱스 추출 및 후보 region 선별
region_ids = plane_signs[:,0]*4 + plane_signs[:,1]*2 + plane_signs[:,2]*1
N_REGIONS = 8
region2idxs = {region: np.where(region_ids == region)[0] for region in range(N_REGIONS)}

# 5개 이하 region은 후보에서 제외, 5개 이상 region 후보만 추출
candidate_regions = [region for region, idxs in region2idxs.items() if len(idxs) >= 5]

# inject_out: (N, C, H, W)
region_coherences = []
for region in candidate_regions:
    idxs = region2idxs[region]
    acts = inject_out[idxs]  # (num, C, H, W)
    # 응집력: 각 feature의 variance의 합(작을수록 응집)
    coherence = -acts.var(dim=0).mean().item()  # 음수: 작을수록 응집, 뒤에서 argmax
    region_coherences.append((region, coherence))

if region_coherences:
    # 가장 응집력 높은 region 번호 선택
    best_region, best_coherence = max(region_coherences, key=lambda t: t[1])
    mask_region = region_ids == best_region
    print(f"가장 응집력 높은 region은 {best_region}번 (coherence={best_coherence:.4f}) 입니다.")
else:
    # 아무 후보 region이 없으면 모두 False
    mask_region = np.zeros(len(region_ids), dtype=bool)
    print("응집력 기준을 만족하는 후보 region이 없습니다.")
pos = np.where(mask_region)[0]
neg = np.where(~mask_region)[0]
print(f"planes {STEER_PLANES}:  (+) {len(pos)} imgs (특정영역)   (-) {len(neg)} imgs (그외)")

act_pos = inject_out[pos].mean(dim=0) if len(pos) > 0 else torch.zeros_like(inject_out[0])
act_neg = inject_out[neg].mean(dim=0) if len(neg) > 0 else torch.zeros_like(inject_out[0])
concept_direction = act_pos - act_neg                 # (+) 영역으로 미는 벡터
# concept_direction = concept_direction / concept_direction.norm()
print(f"concept_direction: {tuple(concept_direction.shape)}  "
      f"|d|={concept_direction.norm():.3f}")

In [ ]:
# ── steering: diverse prompts · original (5) | steered (5) ────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

TEST_PROMPTS = [
    "a photo of a sitting dog",
]
STEER_SEED_BASE = 1100
NUM_SEEDS       = 5
ALPHA_ORIG      = 0
ALPHA_STEER     = 2
SAVE_FIG        = True
FIG_DPI         = 200

FIG_BG = '#FFFFFF'
GRID   = '#CCCCCC'
TEXT   = '#222222'
MUTED  = '#666666'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica Neue', 'DejaVu Sans'],
    'text.color': TEXT,
    'figure.facecolor': FIG_BG,
    'axes.facecolor': FIG_BG,
})

seeds = list(range(STEER_SEED_BASE, STEER_SEED_BASE + NUM_SEEDS))
n_prompts = len(TEST_PROMPTS)

# ── generate: (prompt_idx, alpha, seed) ─────────────────────────────────────
grid_imgs = {}
total = n_prompts * len(seeds) * 2
done  = 0
for pi, prompt in enumerate(TEST_PROMPTS):
    for seed in seeds:
        for alpha in (ALPHA_ORIG, ALPHA_STEER):
            grid_imgs[(pi, alpha, seed)] = generate_with_concept_direction(
                prompt, seed, concept_direction, alpha)
            done += 1
            print(f"[{done}/{total}]  {prompt[:28]:28s}  a={alpha}  s={seed}", end='\r')
print('\nGeneration complete.' + ' ' * 40)

In [ ]:
# ── layout ────────────────────────────────────────────────────────────────────
import textwrap
from matplotlib.patches import FancyBboxPatch

def _prompt_lines(pi, prompt, wrap=58):
    letter = chr(ord('a') + pi)
    quoted = f'"{prompt}"'
    if len(quoted) <= wrap:
        return f'({letter})   {quoted}'
    body = textwrap.fill(quoted, width=wrap, subsequent_indent=' ' * 7)
    return f'({letter})   {body}'

cell     = 1.14
prompt_h = 0.36   # inch per prompt ribbon
hdr_h    = 0.24
row_h    = prompt_h + cell
fig_w    = 2 * NUM_SEEDS * cell + 0.55
fig_h    = hdr_h + n_prompts * row_h + 0.22

fig = plt.figure(figsize=(fig_w, fig_h), facecolor=FIG_BG)
outer = GridSpec(
    1 + n_prompts, 1, figure=fig,
    height_ratios=[hdr_h / row_h] + [1.0] * n_prompts,
    hspace=0.10, left=0.04, right=0.99, top=0.97, bottom=0.03)

# ── global header: Original | Steered ─────────────────────────────────────────
hdr = outer[0].subgridspec(1, 2, wspace=0.055)
for bi, title in enumerate(('Original', 'Steered')):
    ax_h = fig.add_subplot(hdr[0, bi])
    ax_h.axis('off')
    ax_h.text(0.5, 0.72, title, ha='center', va='center',
              fontsize=14, color=TEXT, ) # fontweight='bold'
    pos = ax_h.get_position(fig)
    step = pos.width / NUM_SEEDS
    # for j in range(NUM_SEEDS):
    #     fig.text(pos.x0 + (j + 0.5) * step, pos.y0 + 0.006,
    #              str(j + 1), ha='center', va='bottom',
    #              fontsize=7.5, color=MUTED)

# ── per-prompt block ──────────────────────────────────────────────────────────
PROMPT_BG = '#F4F5F7'
PROMPT_BD = '#E2E5EA'

for pi, prompt in enumerate(TEST_PROMPTS):
    block = outer[pi + 1].subgridspec(
        2, 1, height_ratios=[prompt_h / cell, 1.0], hspace=0.03)

    # prompt ribbon
    ax_p = fig.add_subplot(block[0])
    ax_p.set_xlim(0, 1); ax_p.set_ylim(0, 1)
    ax_p.axis('off')
    panel = ax_p.get_position(fig)
    fig.add_artist(FancyBboxPatch(
        (panel.x0, panel.y0), panel.width, panel.height,
        boxstyle='square,pad=0', linewidth=0.6,
        edgecolor=PROMPT_BD, facecolor=PROMPT_BG,
        transform=fig.transFigure, clip_on=False, zorder=0))
    ax_p.text(
        0.015, 0.5, _prompt_lines(pi, prompt),
        ha='left', va='center', fontsize=12, color=TEXT,
        fontstyle='italic', linespacing=1.35, zorder=1)

    # image row: original | steered
    img_row = block[1].subgridspec(1, 2, wspace=0.055)
    for bi, alpha in enumerate((ALPHA_ORIG, ALPHA_STEER)):
        inner = img_row[0, bi].subgridspec(1, NUM_SEEDS, wspace=0.010)
        for j, seed in enumerate(seeds):
            ax = fig.add_subplot(inner[0, j])
            ax.imshow(np.asarray(grid_imgs[(pi, alpha, seed)]),
                      interpolation='lanczos')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor(GRID)
                sp.set_linewidth(0.5)

    p_o = img_row[0, 0].get_position(fig)
    p_s = img_row[0, 1].get_position(fig)
    fig.add_artist(plt.Line2D(
        [(p_o.x1 + p_s.x0) / 2] * 2, [p_o.y0 + 0.004, p_o.y1 - 0.004],
        transform=fig.transFigure, color=GRID, lw=0.8, zorder=5))

out_name = 'results_steering_multiprompt.png'
if SAVE_FIG:
    fig.savefig(out_name, dpi=FIG_DPI, bbox_inches='tight',
                facecolor=FIG_BG, pad_inches=0.04)
    print(f'saved: {out_name}')
plt.show()